In [1]:
import nba_api.stats.endpoints
import requests
import json
import pandas as pd
import nba_api

In [2]:
# Get Timberwolves player game logs for the season
wolves_player_logs = nba_api.stats.endpoints.PlayerGameLogs(season_nullable='2025-26',team_id_nullable='1610612750').get_data_frames()[0]

wolves_player_logs

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,...,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT
0,2025-26,203944,Julius Randle,Julius,1610612750,MIN,Minnesota Timberwolves,0042500165,2026-04-27T00:00:00,MIN @ DEN,...,1,59,986,87,68,5,53,1,32:09,1
1,2025-26,1630245,Ayo Dosunmu,Ayo,1610612750,MIN,Minnesota Timberwolves,0042500165,2026-04-27T00:00:00,MIN @ DEN,...,466,186,994,329,68,5,267,1,38:09,1
2,2025-26,1629675,Naz Reid,Naz,1610612750,MIN,Minnesota Timberwolves,0042500165,2026-04-27T00:00:00,MIN @ DEN,...,702,369,414,513,68,5,430,1,24:06,1
3,2025-26,1630538,Bones Hyland,Bones,1610612750,MIN,Minnesota Timberwolves,0042500165,2026-04-27T00:00:00,MIN @ DEN,...,122,272,665,571,68,5,446,1,23:13,1
4,2025-26,1630183,Jaden McDaniels,Jaden,1610612750,MIN,Minnesota Timberwolves,0042500165,2026-04-27T00:00:00,MIN @ DEN,...,35,338,1031,585,68,5,516,1,27:15,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1035,2025-26,204060,Joe Ingles,Joe,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,702,742,443,789,68,5,762,1,16:10,1
1036,2025-26,1642389,Zyon Pullin,Zyon,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,702,802,632,899,68,5,870,1,10:19,1
1037,2025-26,1631262,Jules Bernard,Jules,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,702,892,665,921,68,5,926,1,4:10,1
1038,2025-26,1641803,Tristen Newton,Tristen,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,702,802,443,943,68,5,926,1,3:08,1


In [3]:
# Filter for Anthony Edwards and Bones Hyland game logs
edwards_logs = wolves_player_logs[wolves_player_logs['PLAYER_NAME'] == 'Anthony Edwards'].copy()
hyland_logs = wolves_player_logs[wolves_player_logs['PLAYER_NAME'] == 'Bones Hyland'].copy()

print(f"Anthony Edwards games: {len(edwards_logs)}")
print(f"Bones Hyland games: {len(hyland_logs)}")

# Get unique Game IDs for each player
edwards_game_ids = set(edwards_logs['GAME_ID'].unique())
hyland_game_ids = set(hyland_logs['GAME_ID'].unique())

print(f"\nEdwards Game IDs: {len(edwards_game_ids)}")
print(f"Hyland Game IDs: {len(hyland_game_ids)}")

# Find games where Hyland played but Edwards did NOT play
hyland_without_edwards_game_ids = hyland_game_ids - edwards_game_ids

print(f"\nGames where Bones Hyland played WITHOUT Anthony Edwards: {len(hyland_without_edwards_game_ids)}")

Anthony Edwards games: 68
Bones Hyland games: 82

Edwards Game IDs: 68
Hyland Game IDs: 82

Games where Bones Hyland played WITHOUT Anthony Edwards: 24


In [4]:
# Filter Hyland's logs for games where Edwards did NOT play
hyland_without_edwards = hyland_logs[hyland_logs['GAME_ID'].isin(hyland_without_edwards_game_ids)].copy()

print(f"Bones Hyland game logs when Anthony Edwards did NOT play: {len(hyland_without_edwards)}")

if len(hyland_without_edwards) > 0:
    # Sort by date
    hyland_without_edwards = hyland_without_edwards.sort_values('GAME_DATE', ascending=False)
    
    # Display the game logs
    display_cols = ['GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 
                   'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 
                   'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
    display_cols = [col for col in display_cols if col in hyland_without_edwards.columns]
    
    hyland_without_edwards[display_cols]
else:
    print("No games found where Bones Hyland played without Anthony Edwards")

Bones Hyland game logs when Anthony Edwards did NOT play: 24


In [5]:
# Calculate averages for Bones Hyland when Anthony Edwards does NOT play
if len(hyland_without_edwards) > 0:
    # Select numeric columns for averaging
    numeric_cols = ['MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
                   'FTM', 'FTA', 'FT_PCT', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
    
    # Filter to only columns that exist in the dataframe
    numeric_cols = [col for col in numeric_cols if col in hyland_without_edwards.columns]
    
    # Calculate averages
    averages = hyland_without_edwards[numeric_cols].mean()
    
    print("Bones Hyland Averages when Anthony Edwards does NOT play:")
    print("=" * 60)
    
    # Format and display averages
    for col in numeric_cols:
        if col in averages.index:
            value = averages[col]
            if 'PCT' in col:
                print(f"{col:15s}: {value:.3f} ({value*100:.1f}%)")
            else:
                print(f"{col:15s}: {value:.2f}")
    
    print("\n" + "=" * 60)
    print(f"Total games: {len(hyland_without_edwards)}")
    
    # Calculate win-loss record
    if 'WL' in hyland_without_edwards.columns:
        wins = (hyland_without_edwards['WL'] == 'W').sum()
        losses = (hyland_without_edwards['WL'] == 'L').sum()
        if wins + losses > 0:
            win_pct = wins / (wins + losses)
            print(f"Team Record: {wins}-{losses} ({win_pct:.3f} / {win_pct*100:.1f}%)")
else:
    print("No games found to calculate averages")

Bones Hyland Averages when Anthony Edwards does NOT play:
MIN            : 21.27
PTS            : 11.62
FGM            : 3.88
FGA            : 8.79
FG_PCT         : 0.431 (43.1%)
FG3M           : 2.21
FG3A           : 5.71
FG3_PCT        : 0.392 (39.2%)
FTM            : 1.67
FTA            : 2.08
FT_PCT         : 0.412 (41.2%)
REB            : 2.38
AST            : 3.25
STL            : 0.75
BLK            : 0.21
TOV            : 1.62
PF             : 2.17
PLUS_MINUS     : 6.33

Total games: 24
Team Record: 13-11 (0.542 / 54.2%)


In [6]:
# Compare: Hyland's averages WITH vs WITHOUT Edwards
print("Comparison: Bones Hyland WITH vs WITHOUT Anthony Edwards")
print("=" * 70)

# Calculate Hyland's averages in ALL games (including with Edwards)
hyland_all_games = hyland_logs.copy()
hyland_with_edwards = hyland_logs[hyland_logs['GAME_ID'].isin(edwards_game_ids)].copy()

if len(hyland_without_edwards) > 0 and len(hyland_with_edwards) > 0:
    numeric_cols = ['MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
                   'FTM', 'FTA', 'FT_PCT', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
    numeric_cols = [col for col in numeric_cols if col in hyland_logs.columns]
    
    avg_without = hyland_without_edwards[numeric_cols].mean()
    avg_with = hyland_with_edwards[numeric_cols].mean()
    avg_all = hyland_all_games[numeric_cols].mean()
    
    print(f"\n{'Stat':<15} {'WITHOUT Edwards':<20} {'WITH Edwards':<20} {'ALL Games':<20}")
    print("-" * 70)
    
    for col in numeric_cols:
        if col in avg_without.index and col in avg_with.index:
            without_val = avg_without[col]
            with_val = avg_with[col]
            all_val = avg_all[col]
            
            if 'PCT' in col:
                print(f"{col:<15} {without_val:>6.3f} ({without_val*100:>5.1f}%){'':<6} {with_val:>6.3f} ({with_val*100:>5.1f}%){'':<6} {all_val:>6.3f} ({all_val*100:>5.1f}%)")
            else:
                print(f"{col:<15} {without_val:>6.2f}{'':<13} {with_val:>6.2f}{'':<13} {all_val:>6.2f}")
    
    print("\n" + "=" * 70)
    print(f"Games WITHOUT Edwards: {len(hyland_without_edwards)}")
    print(f"Games WITH Edwards: {len(hyland_with_edwards)}")
    print(f"Total games: {len(hyland_all_games)}")
else:
    print(f"\nGames WITHOUT Edwards: {len(hyland_without_edwards)}")
    print(f"Games WITH Edwards: {len(hyland_with_edwards)}")
    if len(hyland_without_edwards) == 0:
        print("\nNo games found where Bones Hyland played without Anthony Edwards")

Comparison: Bones Hyland WITH vs WITHOUT Anthony Edwards

Stat            WITHOUT Edwards      WITH Edwards         ALL Games           
----------------------------------------------------------------------
MIN              21.27               14.43               16.43
PTS              11.62                7.14                8.45
FGM               3.88                2.55                2.94
FGA               8.79                5.60                6.54
FG_PCT           0.431 ( 43.1%)        0.412 ( 41.2%)        0.418 ( 41.8%)
FG3M              2.21                1.40                1.63
FG3A              5.71                3.55                4.18
FG3_PCT          0.392 ( 39.2%)        0.346 ( 34.6%)        0.359 ( 35.9%)
FTM               1.67                0.64                0.94
FTA               2.08                0.79                1.17
FT_PCT           0.412 ( 41.2%)        0.303 ( 30.3%)        0.335 ( 33.5%)
REB               2.38                1.60                1.

In [7]:
# Display Bones Hyland's complete game logs when Anthony Edwards does NOT play
from IPython.display import display

print("Bones Hyland Game Logs (Games where Anthony Edwards did NOT play):")
print("=" * 100)

if len(hyland_without_edwards) > 0:
    # Ensure sorted by date (most recent first)
    hyland_without_edwards_display = hyland_without_edwards.sort_values('GAME_DATE', ascending=False).copy()
    
    # Select columns to display
    display_cols = ['GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 
                   'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 
                   'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
    display_cols = [col for col in display_cols if col in hyland_without_edwards_display.columns]
    
    # Display the dataframe
    display(hyland_without_edwards_display[display_cols])
else:
    print("No games found where Bones Hyland played without Anthony Edwards")

Bones Hyland Game Logs (Games where Anthony Edwards did NOT play):


,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,...,FTM,FTA,FT_PCT,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
3,2026-04-27T00:00:00,MIN @ DEN,L,23.221667,15,5,9,0.556,3,7,...,2,2,1.000,1,1,1,0,3,1,-3
82,2026-04-08T00:00:00,MIN @ ORL,L,22.700000,9,3,8,0.375,3,7,...,0,0,0.000,1,0,1,1,5,1,-6
86,2026-04-07T00:00:00,MIN @ IND,W,20.665000,19,5,9,0.556,4,7,...,5,6,0.833,4,7,1,0,0,3,15
99,2026-04-05T00:00:00,MIN vs. CHA,L,28.683333,18,5,11,0.455,5,11,...,3,4,0.750,6,6,2,1,2,1,-1
128,2026-04-02T00:00:00,MIN @ DET,L,19.166667,6,3,12,0.250,0,6,...,0,0,0.000,3,2,0,0,1,2,9
149,2026-03-28T00:00:00,MIN vs. DET,L,25.983333,6,2,10,0.200,2,9,...,0,0,0.000,1,0,2,0,2,1,-17
160,2026-03-25T00:00:00,MIN vs. HOU,W,29.590000,8,3,11,0.273,2,7,...,0,0,0.000,0,8,0,1,1,0,-1
165,2026-03-22T00:00:00,MIN @ BOS,W,29.433333,23,8,14,0.571,3,7,...,4,4,1.000,3,3,1,0,1,1,26
181,2026-03-20T00:00:00,MIN vs. POR,L,22.195000,17,5,11,0.455,3,7,...,4,4,1.000,0,0,2,0,2,3,-5
188,2026-03-18T00:00:00,MIN vs. UTA,W,24.933333,18,6,12,0.500,3,7,...,3,3,1.000,3,2,0,0,3,2,30


In [8]:
# Calculate Timberwolves record when Bones Hyland scores 10+ points
from IPython.display import display

# Filter for games where Bones Hyland scored 10 or more points
hyland_10plus_pts = hyland_logs[hyland_logs['PTS'] >= 15].copy()

print("Timberwolves Record when Bones Hyland scores 15+ points:")
print("=" * 70)

if len(hyland_10plus_pts) > 0:
    wins = (hyland_10plus_pts['WL'] == 'W').sum()
    losses = (hyland_10plus_pts['WL'] == 'L').sum()
    total = wins + losses
    
    if total > 0:
        win_pct = wins / total
        print(f"\nGames where Bones Hyland scored 15+ points: {total}")
        print(f"Wins: {wins}")
        print(f"Losses: {losses}")
        print(f"Win Percentage: {win_pct:.3f} ({win_pct*100:.1f}%)")
        
        # Show game details
        print(f"\nGame Details:")
        display_cols = ['GAME_DATE', 'MATCHUP', 'WL', 'PTS', 'MIN', 'FGM', 'FGA', 'FG_PCT', 
                       'FG3M', 'FG3A', 'AST', 'REB', 'PLUS_MINUS']
        display_cols = [col for col in display_cols if col in hyland_10plus_pts.columns]
        
        hyland_10plus_display = hyland_10plus_pts[display_cols].sort_values('GAME_DATE', ascending=False)
        display(hyland_10plus_display)
        
        # Compare to overall record
        print(f"\nComparison:")
        print(f"Total games played by Bones Hyland: {len(hyland_logs)}")
        overall_wins = (hyland_logs['WL'] == 'W').sum()
        overall_losses = (hyland_logs['WL'] == 'L').sum()
        overall_total = overall_wins + overall_losses
        if overall_total > 0:
            overall_win_pct = overall_wins / overall_total
            print(f"Overall record when Bones Hyland plays: {overall_wins}-{overall_losses} ({overall_win_pct:.3f} / {overall_win_pct*100:.1f}%)")
    else:
        print("Could not determine win/loss record")
else:
    print("No games found where Bones Hyland scored 10+ points")

Timberwolves Record when Bones Hyland scores 15+ points:

Games where Bones Hyland scored 15+ points: 15
Wins: 10
Losses: 5
Win Percentage: 0.667 (66.7%)

Game Details:


,GAME_DATE,MATCHUP,WL,PTS,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,AST,REB,PLUS_MINUS
3,2026-04-27T00:00:00,MIN @ DEN,L,15,23.221667,5,9,0.556,3,7,1,1,-3
86,2026-04-07T00:00:00,MIN @ IND,W,19,20.665000,5,9,0.556,4,7,7,4,15
99,2026-04-05T00:00:00,MIN vs. CHA,L,18,28.683333,5,11,0.455,5,11,6,6,-1
112,2026-04-03T00:00:00,MIN @ PHI,L,21,30.150000,7,14,0.500,5,9,3,5,-2
165,2026-03-22T00:00:00,MIN @ BOS,W,23,29.433333,8,14,0.571,3,7,3,3,26
181,2026-03-20T00:00:00,MIN vs. POR,L,17,22.195000,5,11,0.455,3,7,0,0,-5
188,2026-03-18T00:00:00,MIN vs. UTA,W,18,24.933333,6,12,0.500,3,7,2,3,30
200,2026-03-17T00:00:00,MIN vs. PHX,W,22,28.875000,8,14,0.571,4,8,5,2,18
293,2026-03-01T00:00:00,MIN @ DEN,W,18,15.590000,6,7,0.857,3,3,2,2,0
375,2026-02-06T00:00:00,MIN vs. NOP,L,20,30.416667,6,11,0.545,4,8,2,3,-5



Comparison:
Total games played by Bones Hyland: 82
Overall record when Bones Hyland plays: 47-35 (0.573 / 57.3%)


In [11]:
# Terrence Shannon Jr. (TJ Shannon) — average stats by minute threshold
# 2024-25 & 2025-26, Regular Season + Playoffs (GAME_ID types 2 & 4 only)
from IPython.display import display
from nba_api.stats.static import players as nba_players
from nba_api.stats.endpoints import PlayerGameLogs

numeric_cols = ['MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
                'FTM', 'FTA', 'FT_PCT', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']

matches = nba_players.find_players_by_full_name('Terrence Shannon Jr.')
if len(matches) != 1:
    raise ValueError(
        f"Expected exactly one Terrence Shannon Jr.; got {len(matches)}. "
        "Set player_id_shannon = 1630545 and re-run if needed."
    )
player_id_shannon = matches[0]['id']

frames = []
for season in ('2024-25', '2025-26'):
    for season_type in ('Regular Season', 'Playoffs'):
        df = PlayerGameLogs(
            player_id_nullable=player_id_shannon,
            season_nullable=season,
            season_type_nullable=season_type,
            timeout=90,
        ).get_data_frames()[0]
        if len(df) > 0:
            frames.append(df)

shannon_logs = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if len(shannon_logs) > 0:
    shannon_logs = shannon_logs[
        shannon_logs['GAME_ID'].astype(str).str[2].isin(['2', '4'])
    ].copy()
    shannon_logs = shannon_logs.drop_duplicates(subset=['GAME_ID'], keep='first')

if 'NBA_FANTASY_PTS' in shannon_logs.columns:
    numeric_cols = numeric_cols + ['NBA_FANTASY_PTS']
numeric_cols = [c for c in numeric_cols if c in shannon_logs.columns]

thresholds = [15, 20, 25, 30]
rows = []
for thr in thresholds:
    sub = shannon_logs[shannon_logs['MIN'] >= thr]
    gp = len(sub)
    row = {'min_threshold': f'>={thr} min', 'GP': gp}
    if gp == 0:
        for c in numeric_cols:
            row[c] = float('nan')
    else:
        for c, v in sub[numeric_cols].mean().items():
            row[c] = v
    rows.append(row)

summary_df = pd.DataFrame(rows)

pct_cols_summary = [c for c in summary_df.columns if 'PCT' in c]
other_num_cols = [
    c for c in summary_df.columns
    if c not in ('min_threshold', 'GP') and c not in pct_cols_summary
]
for c in pct_cols_summary:
    summary_df[c] = summary_df[c].round(3)
for c in other_num_cols:
    summary_df[c] = summary_df[c].round(1)

print("Terrence Shannon Jr. — per-game averages when he plays at least X minutes")
print("Scope: 2024-25 & 2025-26, Regular Season + Playoffs | all games in sample:", len(shannon_logs))
print("=" * 80)
display(summary_df)

for thr in thresholds:
    sub = shannon_logs[shannon_logs['MIN'] >= thr]
    print(f"\nMIN >= {thr}  |  GP: {len(sub)}")
    print("-" * 60)
    if len(sub) == 0:
        print("(no games)")
        continue
    averages = sub[numeric_cols].mean()
    for col in numeric_cols:
        value = averages[col]
        if 'PCT' in col:
            print(f"{col:15s}: {value:.3f}")
        else:
            print(f"{col:15s}: {value:.1f}")

summary_df.to_csv('~/Downloads/shannon_summary.csv', index=False)


Terrence Shannon Jr. — per-game averages when he plays at least X minutes
Scope: 2024-25 & 2025-26, Regular Season + Playoffs | all games in sample: 86


,min_threshold,GP,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,...,FTA,FT_PCT,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,NBA_FANTASY_PTS
0,>=15 min,22,23.4,12.8,4.4,8.5,0.486,1.3,3.0,0.319,...,3.3,0.672,3.1,2.1,0.7,0.2,1.0,2.1,1.5,21.4
1,>=20 min,12,27.9,16.8,5.7,10.8,0.518,1.7,3.7,0.398,...,4.5,0.733,4.2,3.2,0.6,0.4,1.3,2.2,4.8,28.3
2,>=25 min,11,28.6,17.3,5.8,11.2,0.513,1.7,3.7,0.404,...,4.7,0.709,4.2,2.9,0.5,0.4,1.4,2.4,5.2,28.0
3,>=30 min,3,32.1,22.7,6.7,13.0,0.516,1.7,4.7,0.238,...,9.3,0.843,3.3,4.3,1.0,0.3,2.3,2.3,6.3,34.8



MIN >= 15  |  GP: 22
------------------------------------------------------------
MIN            : 23.4
PTS            : 12.8
FGM            : 4.4
FGA            : 8.5
FG_PCT         : 0.486
FG3M           : 1.3
FG3A           : 3.0
FG3_PCT        : 0.319
FTM            : 2.8
FTA            : 3.3
FT_PCT         : 0.672
REB            : 3.1
AST            : 2.1
STL            : 0.7
BLK            : 0.2
TOV            : 1.0
PF             : 2.1
PLUS_MINUS     : 1.5
NBA_FANTASY_PTS: 21.4

MIN >= 20  |  GP: 12
------------------------------------------------------------
MIN            : 27.9
PTS            : 16.8
FGM            : 5.7
FGA            : 10.8
FG_PCT         : 0.518
FG3M           : 1.7
FG3A           : 3.7
FG3_PCT        : 0.398
FTM            : 3.8
FTA            : 4.5
FT_PCT         : 0.733
REB            : 4.2
AST            : 3.2
STL            : 0.6
BLK            : 0.4
TOV            : 1.3
PF             : 2.2
PLUS_MINUS     : 4.8
NBA_FANTASY_PTS: 28.3

MIN >= 25  |  GP